In [1]:
from pyspark.sql import SparkSession

spark = (
        SparkSession.builder
        .appName("ecommerce-streaming")
        .config(
            "spark.sql.catalog.local",
            "org.apache.iceberg.spark.SparkCatalog"
        )
        .config(
            "spark.sql.catalog.local.type",
            "hadoop"
        )
        .config(
            "spark.sql.catalog.local.warehouse",
            "/home/jovyan/work/warehouse"
        )
        .config(
            "spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
        )
        .config(
            "spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1"
        )
        .getOrCreate()
    )

spark

In [37]:
df1 = spark.read.format("iceberg").load("local.bronze.customer_events")
df2 = spark.read.format("iceberg").load("local.silver.customer_events")

In [38]:
print(df1.count())
df2.count()

3181


1828

In [39]:
df1.printSchema()
df2.printSchema()

root
 |-- raw_value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)

root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- source_topic: string (nullable = true)
 |-- source_partition: integer (nullable = true)
 |-- source_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- bronze_ingestion_time: timestamp (nullable = true)
 |-- event_date: date (nullable = true)



In [40]:
from pyspark.sql.functions import col,from_json
from pyspark.sql.types import *

schema = df2.schema

df1_parsed = (
    df1.
    selectExpr("CAST(raw_value as STRING) as json")
    .select(from_json(col("json"),schema).alias("data"))
    .select("data.*")
)

# df2_parsed = (
#     df2.
#     selectExpr("CAST(value as STRING) as json")
#     .select(from_json(col("json"),schema).alias("data"))
#     .select("data.*")
# )

In [41]:
# df1_parsed.printSchema()
df1 = df1_parsed
df1.printSchema()
df2.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- source_topic: string (nullable = true)
 |-- source_partition: integer (nullable = true)
 |-- source_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- bronze_ingestion_time: timestamp (nullable = true)
 |-- event_date: date (nullable = true)

root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: st

In [44]:
missing_records = df1.subtract(df2)

missing_records.select('customer_id').limit(10).show()

+-----------+
|customer_id|
+-----------+
|  cust_1314|
|  cust_2322|
|  cust_4117|
|  cust_7508|
|  cust_2341|
|  cust_7420|
|  cust_1652|
|  cust_9547|
|  cust_6049|
|  cust_3253|
+-----------+



In [45]:
df2.filter(col('customer_id')=='cust_1314').show()

+--------------------+-------------------+--------------------+-----------+----------+---------+--------------------+---------+---------------+---------------+----------------+-------------+--------------------+---------------------+----------+
|            event_id|         event_type|          event_time|customer_id|first_name|last_name|               email|  country|           city|   source_topic|source_partition|source_offset|     kafka_timestamp|bronze_ingestion_time|event_date|
+--------------------+-------------------+--------------------+-----------+----------+---------+--------------------+---------+---------------+---------------+----------------+-------------+--------------------+---------------------+----------+
|5b03355f-7211-446...|customer_registered|2026-06-08 12:25:...|  cust_1314|     Lydia|   Chavez|   krowe@example.com|     OMAN|     Hudsonside|customer_events|               0|         1335|2026-06-08 12:25:...| 2026-06-08 12:25:...|2026-06-08|
|c9bd07bb-825d-4b6..

In [46]:
df1.filter(col('customer_id')=='cust_1314').show()

+--------------------+-------------------+--------------------+-----------+----------+---------+--------------------+---------+---------------+------------+----------------+-------------+---------------+---------------------+----------+
|            event_id|         event_type|          event_time|customer_id|first_name|last_name|               email|  country|           city|source_topic|source_partition|source_offset|kafka_timestamp|bronze_ingestion_time|event_date|
+--------------------+-------------------+--------------------+-----------+----------+---------+--------------------+---------+---------------+------------+----------------+-------------+---------------+---------------------+----------+
|5b03355f-7211-446...|customer_registered|2026-06-08 12:25:...|  cust_1314|     Lydia|   Chavez|   krowe@example.com|     Oman|     Hudsonside|        NULL|            NULL|         NULL|           NULL|                 NULL|      NULL|
|3adf95f9-168c-4f8...|customer_registered|2026-06-03